# 01 · Raw to Bronze

Carrega o `fraudTrain.csv` do Cloud Storage para uma tabela gerenciada no
BigQuery. Espelho fiel do arquivo: campos de negócio todos como `STRING`, mais
dois metadados de governança.

**O `fraudTest.csv` não entra aqui.** Ele fica lacrado em `gs://BUCKET/holdout/`
até outubro, quando vira o *futuro* que o modelo classifica sem nunca ter visto.

---

### Onde o trabalho acontece

O BigQuery lê o arquivo do GCS **direto**, sem passar por aqui. O notebook manda
a ordem e recebe a confirmação — o Colab nunca vê os 351 MB.

O pandas aparece só para ler uma amostra de 5 mil linhas e conferir o formato
antes de disparar a carga.

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.cloud import storage

In [ ]:
!pip install --quiet --upgrade google-cloud-bigquery google-cloud-storage db-dtypes gcsfs


In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Autenticado no Colab — use a conta dona do projeto")
except ImportError:
    print("Fora do Colab: usando as credenciais do ambiente")

## CONFIGURAÇÃO DOS PARÂMETROS:

In [ ]:
PROJECT_ID = "fraudflow-pdm-gps"
BUCKET_NAME = "fraudflow-pdm-gps-data"
ARQUIVO_TRAIN = "raw/train/fraudTrain.csv"
ARQUIVO_HOLDOUT = "holdout/fraudTest.csv"

GCS_URI      = f"gs://{BUCKET_NAME}/{ARQUIVO_TRAIN}"
TABELA_STG    = f"{PROJECT_ID}.bronze.transactions_stg"
TABELA_BRONZE = f"{PROJECT_ID}.bronze.transactions"

client = bigquery.Client(project=PROJECT_ID)
gcs = storage.Client(project=PROJECT_ID)

print(f"Origem  : {GCS_URI}")
print(f"Destino : {TABELA_BRONZE}")

## CONFERÊNCIA DO BUCKET

Só metadados — nenhum arquivo é baixado. O tamanho denuncia troca de lugar: o
treino é o maior.

In [ ]:
objetos = {b.name: b.size for b in gcs.list_blobs(BUCKET_NAME) if b.name.endswith('.csv')}
for nome, tamanho in sorted(objetos.items()):
    print(f"  {nome:<34} {tamanho/1024**2:8.1f} MB")

assert ARQUIVO_TRAIN in objetos, f"faltando {ARQUIVO_TRAIN}"
assert ARQUIVO_HOLDOUT in objetos, f"faltando {ARQUIVO_HOLDOUT}"
assert objetos[ARQUIVO_TRAIN] > objetos[ARQUIVO_HOLDOUT], \
    "o treino deveria ser o MAIOR — train e holdout trocaram de lugar"
print("\nLayout ok: holdout separado")

## SCHEMA DO ARQUIVO DE ORIGEM

O CSV do Sparkov tem 23 colunas, mas a **primeira não tem nome** no cabeçalho — é
o índice que o pandas gravou quando o dataset foi exportado. Declaramos o schema
explicitamente e batizamos a primeira de `row_id`; com `skip_leading_rows=1` o
BigQuery ignora o cabeçalho e mapeia por posição.

As demais mantêm o nome original: a Bronze não renomeia nada.

In [ ]:
COLUNAS_RAW = [
    "row_id", "trans_date_trans_time", "cc_num", "merchant", "category", "amt",
    "first", "last", "gender", "street", "city", "state", "zip", "lat", "long",
    "city_pop", "job", "dob", "trans_num", "unix_time", "merch_lat",
    "merch_long", "is_fraud",
]

# Todos como STRING: e a regra da camada. Se o dado chegou torto, ele fica torto
# aqui, e a decisao de descartar ou corrigir e da Silver.
schema = [bigquery.SchemaField(c, 'STRING') for c in COLUNAS_RAW]
print(f"{len(schema)} colunas declaradas")

### Uma olhada no arquivo antes de carregar

Aqui o pandas é a ferramenta certa: `nrows=5000` baixa alguns megabytes para
conferir o formato. É inspeção, não processamento.

In [ ]:
df_amostra = pd.read_csv(GCS_URI, nrows=5000, dtype=str,
                         header=0, names=COLUNAS_RAW)
print(f"Amostra: {len(df_amostra)} linhas")
print(f"Colunas conferem: {list(df_amostra.columns) == COLUNAS_RAW}")
print(f"Valores de is_fraud: {sorted(df_amostra['is_fraud'].unique())}")
df_amostra.head(3)

## CARGA: O BIGQUERY LÊ O GCS DIRETO

`load_table_from_uri` é uma **ordem**, não uma transferência. O notebook envia
algumas centenas de bytes dizendo *"BigQuery, leia esse arquivo"*, e o serviço
faz a leitura paralelizada do Cloud Storage.

Compare com a alternativa: `pd.read_csv` do bucket baixaria 351 MB até aqui, o
pandas faria o parse em um núcleo, e o `load_table_from_dataframe` mandaria tudo
de volta. Duas travessias de rede para copiar um arquivo entre dois serviços que
já conversam entre si.

Como o job de carga não consegue gravar colunas calculadas, ele escreve numa
tabela de estágio e a próxima célula materializa a Bronze com os metadados.

In [ ]:
job_config = bigquery.LoadJobConfig(
    schema=schema,
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    field_delimiter=',',
    allow_quoted_newlines=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_uri(GCS_URI, TABELA_STG, job_config=job_config)
load_job.result()

print(f"Lidos do GCS : {load_job.input_file_bytes/1024**2:,.1f} MB")
print(f"Linhas        : {load_job.output_rows:,}")
print(f"Trafegou pelo Colab: 0 bytes de dado")

## Metadados de governança

`source_file` permite provar de qual arquivo cada linha veio — é o que sustenta,
na apresentação, que o holdout ficou de fora. `ingestion_timestamp` torna a carga
auditável.

**Sem particionamento**, de propósito: a carga acontece de uma vez só, então
particionar por data de ingestão criaria uma única partição gigante.

In [ ]:
SQL_BRONZE = f"""
CREATE OR REPLACE TABLE `{TABELA_BRONZE}` AS
SELECT
  s.*,
  CURRENT_TIMESTAMP() AS ingestion_timestamp,
  '{GCS_URI}'         AS source_file
FROM `{TABELA_STG}` AS s
"""

job = client.query(SQL_BRONZE)
job.result()

client.delete_table(TABELA_STG, not_found_ok=True)
print(f"Bronze materializada · BigQuery leu {job.total_bytes_processed/1024**2:,.1f} MB")
print("Estagio removido")

## VALIDAÇÃO

Tem que dar exatamente **1.296.675**. Número menor indica cópia truncada: as
versões que circulam fora do Kaggle param em 1.048.575, o limite do Excel.

In [ ]:
tabela = client.get_table(TABELA_BRONZE)
print(f"Linhas   : {tabela.num_rows:,}")
print(f"Tamanho  : {tabela.num_bytes/1024**2:.1f} MB")
print(f"Colunas  : {len(tabela.schema)}")
print(f"Particao : {tabela.time_partitioning}")

assert tabela.num_rows == 1_296_675, f"esperado 1.296.675, veio {tabela.num_rows:,}"
assert tabela.time_partitioning is None, "a Bronze nao deve ser particionada"
print("\nBronze ok")

Confirmação de que só o treino entrou. Esta consulta agrega 1,3 milhão de linhas
no BigQuery e devolve **uma**:

In [ ]:
client.query(f"""
    SELECT source_file, COUNT(*) AS linhas, MIN(ingestion_timestamp) AS carregado_em
    FROM `{TABELA_BRONZE}`
    GROUP BY source_file
""").to_dataframe()

---

**Próximo:** `02_bronze_to_silver.ipynb`